# Verify the `delta_lakehouse` skill against a real Fabric Lakehouse

Everything in this skill was measured against Delta tables built locally with
`delta-rs`. That covers the Delta protocol, but not Fabric: not a real mount,
not `abfss://`, and not tables written by Spark (which produces genuine
deletion vector files, V-Order, and its own file layout).

This notebook closes that gap. It installs the release, pulls the skill's own
code out of the installed Markdown, and runs each claim against your table,
reporting PASS / FAIL / SKIP per claim.

**This notebook is read-only.** It never writes, deletes, compacts, or vacuums
anything. The only write is an optional summary printed to the cell output.

Release under test: `fabric-rlm 0.5.0`

## 1. Install

Install the analytics extra because the skill uses DuckDB and delta-rs.
The package version is pinned so the notebook remains reproducible.

`%pip install` must run before anything imports `fabric_rlm`, and Fabric may
need a session restart afterwards.

In [ ]:
%pip install --quiet "fabric-rlm[analytics]==0.6.0"

To test uncommitted local edits, build a wheel (`python -m build
--wheel`), upload `dist/fabric_rlm-*.whl` to `Files/wheels/`, and install that:

```python
import glob
whl = sorted(glob.glob("/lakehouse/default/Files/wheels/fabric_rlm-*.whl"))[-1]
%pip install --quiet {whl}
```

Fabric may need a session restart before the new version is importable. If the
version below is not the one you just installed, restart the session and re-run
from here (skip the install cell).

In [ ]:
import fabric_rlm
from fabric_rlm.skill_loader import SkillLoader

print("fabric_rlm", fabric_rlm.__version__, "from", fabric_rlm.__file__)
print("skills packaged:", sorted(SkillLoader().list_skills()))
assert "delta_lakehouse" in SkillLoader().list_skills(), \
    "delta_lakehouse is missing - you are on an older build, restart the session"

## 2. Point this at your table

Set these to a table you can read. A **partitioned** table with some delete or
merge history exercises the most claims; a plain table still checks the core
path. Nothing here modifies the table.

In [ ]:
WORKSPACE = "<your workspace name>"      # the workspace the lakehouse lives in
LAKEHOUSE = "<your lakehouse name>"      # without the .Lakehouse suffix
SCHEMA    = None                          # e.g. "dbo" for a schema-enabled lakehouse, else None
TABLE     = "<your table name>"

TEST_MOUNTED = True     # requires the lakehouse to be attached to this notebook
TEST_ABFSS   = True     # works whether or not it is attached

def _is_guid(s):
    parts = str(s).split("-")
    return len(parts) == 5 and len(parts[0]) == 8

# The .Lakehouse suffix belongs on a NAME. Appending it to a GUID produces a
# path that is neither a valid Guid nor a valid Name, and OneLake rejects it
# with 400 FriendlyNameSupportDisabled.
_item = LAKEHOUSE if _is_guid(LAKEHOUSE) else f"{LAKEHOUSE}.Lakehouse"
_rel = f"{SCHEMA}/{TABLE}" if SCHEMA else TABLE
MOUNTED_PATH = f"/lakehouse/default/Tables/{_rel}"
ABFSS_PATH = (
    f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
    f"{_item}/Tables/{_rel}"
)
# Root for attaching every table at once. Point at the whole Tables/ folder,
# or at one schema inside it (".../Tables/dbo").
LAKEHOUSE_ROOT = (
    f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/{_item}/Tables"
)

print("mounted:", MOUNTED_PATH)
print("abfss  :", ABFSS_PATH)
print("root   :", LAKEHOUSE_ROOT)
print("       (lakehouse read as a", "GUID - no .Lakehouse suffix)" if _is_guid(LAKEHOUSE)
      else "name - .Lakehouse appended)")

## 3. Harness

`check` records one result per claim. A claim that cannot be evaluated on your
table (no tombstones, not partitioned, no deletion vectors) reports **SKIP**
with the reason rather than a misleading pass.

In [ ]:
RESULTS = []

def check(name, fn):
    """fn returns (True|False|None, note). None means 'not applicable here'."""
    try:
        ok, note = fn()
    except Exception as e:
        RESULTS.append((name, "FAIL", f"{type(e).__name__}: {str(e)[:600]}"))
        print(f"FAIL  {name}\n      {type(e).__name__}: {str(e)[:200]}")
        return
    status = {True: "PASS", False: "FAIL", None: "SKIP"}[ok]
    RESULTS.append((name, status, note))
    print(f"{status}  {name}" + (f"\n      {note}" if note else ""))

## 3b. Preflight: does that path actually exist?

A wrong table name fails deep inside the Delta reader with a confusing error.
Check it here first, and list what is really available.

In [ ]:
import os, difflib

TABLES_ROOT = "/lakehouse/default/Tables"
print("=== attached lakehouse ===")
if not os.path.isdir(TABLES_ROOT):
    print(f"  {TABLES_ROOT} does not exist: no lakehouse is attached to this notebook.")
    print("  Attach one in the Explorer pane, or set TEST_MOUNTED = False.")
else:
    entries = sorted(os.listdir(TABLES_ROOT))
    print(f"  {TABLES_ROOT}: {len(entries)} entries")
    for e in entries[:25]:
        sub = os.path.join(TABLES_ROOT, e)
        if os.path.isdir(os.path.join(sub, "_delta_log")):
            print(f"    table   {e}")
        elif os.path.isdir(sub):
            print(f"    schema  {e}/ -> {sorted(os.listdir(sub))[:12]}")

print()
print("=== your configured mounted path ===")
if os.path.isdir(os.path.join(MOUNTED_PATH, "_delta_log")):
    print(f"  OK  {MOUNTED_PATH}")
else:
    print(f"  NOT FOUND  {MOUNTED_PATH}")
    parent = os.path.dirname(MOUNTED_PATH)
    if os.path.isdir(parent):
        avail = sorted(os.listdir(parent))
        print(f"  available in {parent}:")
        print(f"    {avail}")
        close = difflib.get_close_matches(os.path.basename(MOUNTED_PATH), avail, n=3, cutoff=0.4)
        if close:
            print(f"  >>> did you mean: {close}")
    else:
        print(f"  parent {parent} does not exist either - check SCHEMA")

print()
print("=== your configured abfss path ===")
try:
    import notebookutils
    parent = ABFSS_PATH.rsplit("/", 1)[0]
    names = [f.name for f in notebookutils.fs.ls(parent)]
    print(f"  {parent}")
    print(f"    {sorted(names)[:25]}")
    leaf = ABFSS_PATH.rsplit("/", 1)[1]
    print(f"  {'OK' if leaf in names else 'NOT FOUND'}  {leaf}")
except Exception as e:
    print(f"  could not list: {type(e).__name__}: {str(e)[:300]}")

## 4. Does the router send a Fabric-shaped question to the skill?

These are the exact question shapes a user would type.

In [ ]:
from fabric_rlm.skill_router import SkillRouter

router = SkillRouter.from_loader(SkillLoader())

def _routes(q):
    d = router.route(q)
    return ("delta_lakehouse" in d.active, f"active={d.active} scores={d.scores}")

for q in [
    f"what are the top 10 rows by value in the {TABLE} delta table",
    f"explore {MOUNTED_PATH} and summarize it",
    f"profile {ABFSS_PATH}",
]:
    check(f"routes: {q[:58]}...", lambda q=q: _routes(q))

# And must NOT fire on unrelated work.
check("does not route a plain csv question", lambda: (
    "delta_lakehouse" not in router.route("count rows in Files/data.csv").active, ""))

## 5. Load the skill's own code out of the installed Markdown

This is the point of the exercise: the functions below are not retyped here,
they are extracted from the skill text that shipped in the wheel. If the skill
is wrong, this cell fails.

In [ ]:
import re

SKILL = SkillLoader().load("delta_lakehouse").content
blocks = re.findall(r"```python\n(.*?)\n```", SKILL, re.DOTALL)

resolver = [b for b in blocks if "def open_delta" in b]
discovery = [b for b in blocks if "=== profile ===" in b]
assert len(resolver) == 1, f"expected 1 open_delta block, found {len(resolver)}"
assert len(discovery) == 1, f"expected 1 discovery block, found {len(discovery)}"

exec(compile(resolver[0], "<delta_lakehouse:open_delta>", "exec"), globals())
DISCOVERY_SRC = discovery[0]
print("loaded open_delta / delta_opts / _storage_token from the installed skill")
print("discovery block:", len(DISCOVERY_SRC.splitlines()), "lines")

## 6. Can the skill open your table, both ways?

In [ ]:
import duckdb

HANDLES = {}

def _open(path, label):
    con, T = open_delta(path)
    HANDLES[label] = (con, T)
    n = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    engine = "delta_scan" if "delta_scan" in T else "delta-rs fallback"
    return (n >= 0, f"{engine}, {n:,} rows")

if TEST_MOUNTED:
    check("open attached lakehouse mount", lambda: _open(MOUNTED_PATH, "mount"))
else:
    check("open attached lakehouse mount", lambda: (None, "TEST_MOUNTED is False"))

if TEST_ABFSS:
    check("open abfss:// OneLake path", lambda: _open(ABFSS_PATH, "abfss"))
else:
    check("open abfss:// OneLake path", lambda: (None, "TEST_ABFSS is False"))

# Both paths must agree on the row count.
def _agree():
    if "mount" not in HANDLES or "abfss" not in HANDLES:
        return (None, "need both paths open")
    a = HANDLES["mount"][0].sql(f"SELECT count(*) FROM {HANDLES['mount'][1]}").fetchone()[0]
    b = HANDLES["abfss"][0].sql(f"SELECT count(*) FROM {HANDLES['abfss'][1]}").fetchone()[0]
    return (a == b, f"mount={a:,} abfss={b:,}")

check("mount and abfss return the same count", _agree)

if not HANDLES:
    PATH, con, T = None, None, None
    print()
    print(">>> Neither path opened. Nothing below can run until this is fixed.")
    print(">>> Re-check section 3b: the table name, SCHEMA, and workspace/lakehouse.")
else:
    label = "mount" if "mount" in HANDLES else "abfss"
    PATH = MOUNTED_PATH if label == "mount" else ABFSS_PATH
    con, T = HANDLES[label]
    print()
    print(f"using the {label} path for the checks below: {PATH}")

## 7. The discovery block, run verbatim

Schema, partitions, sample rows, and a per-column profile. Watch the output
size: the skill claims this is cheap enough to be mandatory on turn 1.

In [ ]:
import io, contextlib

assert con is not None, "No table is open - fix section 6 before running this cell."
buf = io.StringIO()
_ns = dict(globals()); _ns["path"] = PATH
with contextlib.redirect_stdout(buf):
    exec(compile(DISCOVERY_SRC, "<delta_lakehouse:discovery>", "exec"), _ns)

out = buf.getvalue()
print(out)

check("discovery prints all four sections", lambda: (
    all(s in out for s in ("=== table ===", "=== schema ===", "=== sample ===", "=== profile ===")),
    "",
))
check("discovery output stays small enough for turn 1", lambda: (
    len(out) < 4000, f"{len(out)} chars, {out.count(chr(10))} lines"))

## 7a. Find your GUIDs

Many tenants set `FriendlyNameSupportDisabled`, which rejects workspace and
lakehouse **names** in a OneLake URL with:

```
WorkspaceId and ArtifactId should be either valid Guids or valid Names
ErrorCode:FriendlyNameSupportDisabled
```

That reads like the names are malformed. They are not; the tenant simply does
not accept them. GUIDs always work. This cell finds yours.

In [ ]:
import notebookutils

ctx = notebookutils.runtime.context
print("notebookutils.runtime.context keys:")
for k in sorted(ctx):
    if any(t in k.lower() for t in ("id", "name", "workspace", "lakehouse")):
        print(f"  {k} = {ctx[k]}")

try:
    DERIVED_ROOT = onelake_root()
    print()
    print("derived root:", DERIVED_ROOT)
except Exception as e:
    DERIVED_ROOT = None
    print()
    print(f"could not derive: {type(e).__name__}: {e}")
    print("Set LAKEHOUSE_ABFSS by hand in the next cell.")

## 7b. Point at one lakehouse, get every table

Set **one** value: the lakehouse in OneLake. Everything else is derived. Use the
GUID form if 7a said your tenant needs it. With a GUID there is no `.Lakehouse`
suffix; with a name there is.

```
abfss://<workspaceId>@onelake.dfs.fabric.microsoft.com/<lakehouseId>
abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse
```

A trailing `/Tables`, a trailing slash, or a specific schema on the end are all
accepted.

In [ ]:
# Optional override. By default, reuse the configured table's OneLake root;
# if ABFSS testing is disabled, fall back to the attached Lakehouse from 7a.
LAKEHOUSE_ABFSS = None      # e.g. "abfss://<wsId>@onelake.dfs.fabric.microsoft.com/<lhId>"

CONFIGURED_ROOT = ABFSS_PATH.split("/Tables/", 1)[0] if TEST_ABFSS else None
root = (LAKEHOUSE_ABFSS or CONFIGURED_ROOT or DERIVED_ROOT or "").rstrip("/")
assert root, "Set LAKEHOUSE_ABFSS, or run 7a so DERIVED_ROOT is available."
if "/Tables" not in root:
    root += "/Tables"
LAKEHOUSE_ROOT = root
print("lakehouse root:", LAKEHOUSE_ROOT)

## 7c. Attach the whole lakehouse

Instead of one table, register every Delta table under the root as a DuckDB
view, so cross-table questions become ordinary SQL joins. Works on an abfss
root or a mount, and on a single schema.

Attaching binds each view, which reads one `_delta_log` per table over the
network. It reads no data.

In [ ]:
import time

t0 = time.time()
ATTACH_ERROR = None
try:
    lh, attached, skipped = attach_lakehouse(LAKEHOUSE_ROOT)
except Exception as e:
    lh, attached, skipped = None, [], []
    ATTACH_ERROR = f"{type(e).__name__}: {e}"
    print("ATTACH FAILED")
    print(f"  root:  {LAKEHOUSE_ROOT}")
    print(f"  error: {ATTACH_ERROR[:1500]}")
    print()
    print("  If this is a OneLake/abfss problem, the mount needs no ids at all.")
    print("  Set LAKEHOUSE_ABFSS = '/lakehouse/default/Tables' in 7b, re-run 7b, then this cell.")
elapsed = time.time() - t0

print(f"attached {len(attached)} tables in {elapsed:.1f}s")
for name in attached:
    print("   ", name)
if skipped:
    print(f"skipped {len(skipped)}:")
    for name, why in skipped:
        print(f"    {name}: {why}")

check("attach_lakehouse found at least one table",
      lambda: (len(attached) > 0, f"{len(attached)} attached, {len(skipped)} skipped "
                                  f"in {elapsed:.1f}s"))

check("attached objects are views, not copies", lambda: (
    None if lh is None else
    set(dict(lh.sql("SELECT table_name, table_type FROM information_schema.tables")
             .fetchall()).values()) == {"VIEW"},
    "nothing attached" if lh is None else
    "a CTAS would have copied the lakehouse into memory"))

Now query across them. Replace this with a join that means something on your
data; the point is that no per-table setup was needed.

In [ ]:
assert lh is not None, (
    f"Nothing attached from {LAKEHOUSE_ROOT}. "
    f"7c reported -> {ATTACH_ERROR}"
)
print(lh.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    ORDER BY 1, 2
""").fetchall())

# Example: row counts for every attached table, in one pass.
for name in attached[:15]:
    ref = ".".join(f'"{p}"' for p in name.split("."))
    try:
        n = lh.sql(f"SELECT count(*) FROM {ref}").fetchone()[0]
        print(f"  {name:40} {n:>12,} rows")
    except Exception as e:
        print(f"  {name:40} ERROR {type(e).__name__}: {str(e)[:70]}")

## 7d. Which of your tables can actually prove what?

Several claims need a table with specific history: tombstoned files need an
un-vacuumed delete or merge, the deletion-vector checks need that feature on,
and the sampling check needs 2+ partitions. A pristine table reports SKIP,
which is honest but proves nothing.

This surveys everything attached and names the best candidate, so section 8
runs against a table that can actually exercise the claims.

In [ ]:
from deltalake import DeltaTable
import glob as _g, os

survey = []
for qname, path in sorted(find_delta_tables(LAKEHOUSE_ROOT).items()):
    try:
        d = DeltaTable(path, storage_options=delta_opts(path))
        live = len(d.file_uris())
        feats = d.protocol().reader_features or []
        nparts = len(d.partitions())
        phys = (len(_g.glob(os.path.join(path, "**", "*.parquet"), recursive=True))
                if "://" not in path else None)
        try:
            ndv = pa.table(d.deletion_vectors().read_all()).num_rows
        except Exception:
            ndv = 0
        survey.append(dict(name=qname, path=path, ver=d.version(), live=live,
                           disk=phys, parts=nparts, dv=ndv,
                           feats=",".join(feats)))
    except Exception as e:
        survey.append(dict(name=qname, path=path, ver="?", live="?", disk="?",
                           parts="?", dv=0, feats=f"ERR {type(e).__name__}"))

print(f"{'table':32} {'ver':>4} {'live':>5} {'disk':>5} {'parts':>5} {'dv':>3}  features")
for r in survey:
    print(f"  {r['name']:30} {str(r['ver']):>4} {str(r['live']):>5} "
          f"{str(r['disk']):>5} {str(r['parts']):>5} {str(r['dv']):>3}  {r['feats']}")

def _score(r):
    s = 0
    if isinstance(r["disk"], int) and isinstance(r["live"], int) and r["disk"] > r["live"]:
        s += 8                      # tombstones: proves the headline claim
    if r["dv"]:
        s += 6                      # real deletion vectors
    if isinstance(r["parts"], int) and r["parts"] >= 2:
        s += 3                      # partition sampling bias
    if isinstance(r["ver"], int) and r["ver"] > 0:
        s += 2                      # has some history at all
    return s

ranked = sorted(survey, key=_score, reverse=True)
best = ranked[0] if ranked and _score(ranked[0]) > 0 else None
print()
if best:
    print(f"best candidate: {best['name']}  (score {_score(best)})")
    print(f"  {best['path']}")
    print("  Set PATH to this and re-run section 8 to exercise more claims:")
    print(f"  PATH = {best['path']!r}")
else:
    print("No table here has tombstones, deletion vectors, or 2+ partitions.")
    print("Section 8 will SKIP those claims - honest, but they stay unproven on Fabric.")
    print("A table that has had a DELETE or MERGE without a subsequent VACUUM is what")
    print("would exercise the headline claim.")

To use the recommendation, set `PATH` and re-open before running section 8:

```python
PATH = "<paste the path printed above>"
con, T = open_delta(PATH)
```

## 8. The correctness claims

The headline one first: on a table with delete/merge history and no vacuum
since, reading the parquet directly must disagree with the Delta reader. If
your table has never been mutated there is nothing to disagree about, and the
check reports SKIP.

In [ ]:
from deltalake import DeltaTable
import pyarrow as pa, glob as _glob, os

assert con is not None, "No table is open - fix section 6 before running this cell."
dt = DeltaTable(PATH, storage_options=delta_opts(PATH))
live = dt.file_uris()
features = dt.protocol().reader_features or []
parts = dt.partitions()
def _dv_entries(t):
    # How many data files actually carry a deletion vector, not just the flag.
    try:
        return pa.table(t.deletion_vectors().read_all()).num_rows
    except Exception:
        return 0

DV = _dv_entries(dt)
print(f"version={dt.version()}  live files={len(live)}  reader_features={features}  "
      f"partitions={len(parts)}  deletion_vectors={DV}")

def _glob_diverges():
    if "://" in PATH:
        return (None, "physical file listing needs a filesystem path, not abfss")
    physical = _glob.glob(os.path.join(PATH, "**", "*.parquet"), recursive=True)
    if len(physical) <= len(live):
        return (None, f"no tombstoned files present ({len(physical)} on disk, {len(live)} live) - "
                      "table has no un-vacuumed delete/merge history")
    truth = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    wrong = con.sql(
        f"SELECT count(*) FROM read_parquet('{PATH}/**/*.parquet')").fetchone()[0]
    return (wrong != truth,
            f"glob={wrong:,} vs delta={truth:,} across {len(physical)} files on disk / {len(live)} live")

check("read_parquet glob disagrees with the Delta reader", _glob_diverges)


def _file_uris_diverges():
    if "deletionVectors" not in features:
        return (None, "table has no deletionVectors reader feature")
    if DV == 0:
        return (None, "deletionVectors is enabled but no data file carries one yet, "
                      "so there is nothing here for a parquet read to get wrong")
    truth = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    lst = ", ".join(f"'{u}'" for u in live)
    viauris = con.sql(f"SELECT count(*) FROM read_parquet([{lst}])").fetchone()[0]
    return (viauris != truth,
            f"read_parquet(file_uris())={viauris:,} vs delta={truth:,} - "
            "this is the 'looks rigorous but is wrong' case")

check("read_parquet(file_uris()) is wrong under deletion vectors", _file_uris_diverges)


def _delta_rs_refuses():
    if "deletionVectors" not in features:
        return (None, "table has no deletionVectors reader feature")
    from deltalake.exceptions import DeltaProtocolError
    try:
        dt.to_pyarrow_dataset()
        return (False, "to_pyarrow_dataset() succeeded - deltalake may now support DVs, "
                       "in which case the skill's hard-fail branch can be relaxed")
    except DeltaProtocolError as e:
        return (True, f"raised as expected: {str(e)[:110]}")

check("delta-rs refuses deletion-vector tables", _delta_rs_refuses)


def _num_records_upper_bound():
    adds = pa.table(dt.get_add_actions()).to_pydict()
    meta = sum(adds.get("num_records", []))
    actual = con.sql(f"SELECT count(*) FROM {T}").fetchone()[0]
    if DV == 0:
        return (meta == actual,
                f"no deletion vectors present, so metadata should be exact: {meta:,} vs {actual:,}")
    # The case only reasoning covered locally: delta-rs rewrote files rather than
    # emitting a deletion vector, so no local table ever exercised the over-count.
    return (meta >= actual,
            f"metadata={meta:,} actual={actual:,} across {DV} deletion vector(s) - "
            f"{'over-counts as predicted' if meta > actual else 'equal despite DVs'}")

check("num_records is exact without DVs, an upper bound with them", _num_records_upper_bound)


def _limit_is_biased():
    if len(parts) < 2:
        return (None, f"table has {len(parts)} partition(s); need 2+ to show the bias")
    col = list(parts[0].keys())[0]
    lim = {r[0] for r in con.sql(f"SELECT {col} FROM {T} LIMIT 20").fetchall()}
    smp = {r[0] for r in con.sql(f"SELECT {col} FROM {T} USING SAMPLE 200 ROWS").fetchall()}
    return (len(lim) < len(smp),
            f"LIMIT saw {len(lim)} distinct {col}, USING SAMPLE saw {len(smp)}")

check("LIMIT is biased to one partition, USING SAMPLE is not", _limit_is_biased)


def _approx_unique_is_approx():
    rel = con.sql(f"SUMMARIZE SELECT * FROM {T}")
    cols = [d[0] for d in rel.description]
    rows = [dict(zip(cols, r)) for r in rel.fetchall()]
    target = max(rows, key=lambda d: int(d["approx_unique"] or 0))
    name = target["column_name"]
    exact = con.sql(f'SELECT count(DISTINCT "{name}") FROM {T}').fetchone()[0]
    approx = int(target["approx_unique"])
    return (True, f"{name}: approx={approx:,} exact={exact:,} "
                  f"({'differs - quote the exact one' if approx != exact else 'equal here, still an estimate'})")

check("approx_unique is an estimate, not a fact", _approx_unique_is_approx)


def _show_encoding():
    try:
        con.sql(f"SELECT * FROM {T} LIMIT 2").show()
        return (None, "no UnicodeEncodeError on this runtime - the trap is cp1252/Windows "
                      "specific, and fetchall() is portable either way")
    except UnicodeEncodeError as e:
        return (True, f"reproduced: {str(e)[:110]}")

check(".show() encoding trap", _show_encoding)

## 9. Read-only confirmation

The table must be exactly where it started. If this fails, something in the run
wrote to your table and that is a bug worth reporting.

In [ ]:
_before = dt.version()
_after = DeltaTable(PATH, storage_options=delta_opts(PATH)).version()
check("table version unchanged by this notebook",
      lambda: (_before == _after, f"version {_before} -> {_after}"))

## 10. Summary

Paste this output back if anything failed.

In [ ]:
import collections, platform

tally = collections.Counter(s for _, s, _ in RESULTS)
print(f"fabric_rlm {fabric_rlm.__version__} | duckdb {duckdb.__version__} | "
      f"python {platform.python_version()}")
print(f"table: {PATH}")
print(f"reader_features={features} partitions={len(parts)} version={dt.version()}")
print()
for name, status, note in RESULTS:
    print(f"{status:5} {name}")
    if note:
        print(f"      {note}")
print()
print("  ".join(f"{k}={v}" for k, v in sorted(tally.items())))
if tally.get("FAIL"):
    print("\nFAILURES above are real gaps in the skill, not notebook problems.")
assert tally.get("FAIL", 0) == 0, "Delta Lakehouse verification failed"